# Task variance and clusters along the iso-accuracy contour

Four frozen-$W_{out}$ cog20 networks at 150k steps, two seeds each. Three of them sit on the
`edge30` **`hi`** contour — equally competent (min-task accuracy 0.74–0.82) but with a different
cost binding — plus the unconstrained control:

| point | character | metabolic | wiring | conn_frac |
|---|---|---|---|---|
| `control` | no budget | 2.5–4.3 | 0.031–0.042 | 0.66–0.74 |
| `hi0` | rate-limited: cheap firing, dense wiring | 0.030 | 0.0400 | 0.23–0.25 |
| `hi5` | middle | 0.43 | 0.0125 | 0.078–0.092 |
| `hi9` | wiring-limited: hot firing, sparse | 4.4–4.6 | 0.0061 | 0.027–0.030 |

**151× in firing and 8.4× in wire count at constant accuracy** — so any structural difference
between them is about *which cost binds*, not about competence.

Method follows `reports/theory.md` §6 and `results/scripts/fig_task_variance_clusters.py`:
`task_variance` is already stored in each checkpoint (variance across stimulus conditions at each
time step after fixation, averaged over time and over the 10 eval seeds, noise-free), so nothing is
re-simulated here.


## Setup

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.gridspec import GridSpecFromSubplotSpec
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from cmc.paths import MOO_ZOOM_RUNS_DIR
from cmc.task import rule_name

EDGE = MOO_ZOOM_RUNS_DIR / "dale_all_edge30_woutfrozen_150k"
RATE = MOO_ZOOM_RUNS_DIR / "dale_all_rateray_woutfrozen_150k"
POINTS = {"control": (RATE / "control", "unconstrained"),
          "hi0":     (EDGE / "hi0",     "rate-limited  (cheap firing, dense)"),
          "hi5":     (EDGE / "hi5",     "middle"),
          "hi9":     (EDGE / "hi9",     "wiring-limited  (hot, sparse)")}
SEEDS = [0, 1]

K_RANGE     = range(2, 20)   # as in results/scripts/fig_task_variance_clusters.py
ACTIVE_FRAC = 0.01           # a unit is active if its total TV > 1% of the network max
N_NULL      = 5              # shuffles for the chance band
SIL_REF     = 0.2            # reference line asked for in the silhouette panels

ck = {(p, s): torch.load(d / f"seed_{s:02d}.pt", map_location="cpu", weights_only=False)
      for p, (d, _) in POINTS.items() for s in SEEDS}
TASKS = list(ck[("control", 0)]["active_tasks"])
N_E   = int(ck[("control", 0)]["model_kwargs"]["n_neurons"]
            * ck[("control", 0)]["model_kwargs"]["frac_e"])

print(f"{'point':9s} {'seed':>4s} {'min_acc':>8s} {'metabolic':>10s} {'wiring':>9s} {'conn_frac':>10s}")
for p in POINTS:
    for s in SEEDS:
        m = ck[(p, s)]["metrics"]
        print(f"{p:9s} {s:4d} {m['min_task_acc']:8.3f} {m['metabolic_cost']:10.4f} "
              f"{m['wiring_cost']:9.5f} {m['conn_frac']:10.4f}")

## Clustering

Active units clear 1% of the network's max summed task variance; each unit's 20-vector is
normalised by its own max, so clustering compares **which** tasks a unit cares about rather than
how hard it fires. k-means for k = 2…19, and a chance band from shuffling each unit's vector
across tasks (5 shuffles, re-clustered at every k).


In [ ]:
def cluster_units(tv, seed=0, k_range=K_RANGE):
    """Yang-2019 clustering of a (n_tasks, N) task-variance matrix.

    Active units are those whose summed task variance clears 1% of the network max.
    Each unit's vector is normalised by its own max over tasks, so clustering compares
    *which* tasks a unit cares about, not how hard it fires. Returns the active mask,
    the normalised matrix X (n_active, n_tasks), the silhouette score per k, and the
    labels per k.
    """
    total  = tv.sum(axis=0)
    active = total > ACTIVE_FRAC * total.max()
    X = (tv[:, active] / tv[:, active].max(axis=0)).T
    scores, labels = {}, {}
    for k in k_range:
        lab = KMeans(k, n_init=10, random_state=seed).fit_predict(X)
        scores[k], labels[k] = silhouette_score(X, lab), lab
    return active, X, scores, labels


def null_scores(X, n_null=N_NULL, seed=0):
    """Chance silhouette: shuffle each unit's task-variance vector across tasks."""
    rng = np.random.default_rng(seed)
    out = {k: [] for k in K_RANGE}
    for i in range(n_null):
        Xs = np.array([rng.permutation(row) for row in X])
        for k in K_RANGE:
            lab = KMeans(k, n_init=10, random_state=seed).fit_predict(Xs)
            out[k].append(silhouette_score(Xs, lab))
    return {k: (np.min(v), np.max(v)) for k, v in out.items()}


FIT = {}
for p in POINTS:
    for s in SEEDS:
        tv = np.asarray(ck[(p, s)]["task_variance"], dtype=float)
        active, X, scores, labels = cluster_units(tv, seed=s)
        k_best = max(scores, key=scores.get)
        FIT[(p, s)] = dict(active=active, X=X, scores=scores, labels=labels, k=k_best,
                           null=null_scores(X, seed=s))
        print(f"{p:9s} seed {s}:  active {int(active.sum()):3d}/{tv.shape[1]}   "
              f"best k = {k_best:2d}   silhouette {scores[k_best]:.3f}   "
              f"(null at that k: {FIT[(p,s)]['null'][k_best][0]:.3f}-{FIT[(p,s)]['null'][k_best][1]:.3f})")

## Seed 0

One row per network. **Left:** normalised task variance, units sorted by cluster (clusters ordered
by their dominant task), with cluster and E/I strips underneath. **Right:** silhouette across k,
the shuffled-null band, the selected k circled, and a dashed reference line at 0.2.


In [ ]:
INK, MUTED, SURF, ALERT = "#1a1a19", "#5c5b55", "#fcfcfb", "#e34948"

def panel_row(fig, spec, p, s):
    """One point: task-variance map (left) + silhouette vs k (right)."""
    f = FIT[(p, s)]
    X, labels, k = f["X"], f["labels"][f["k"]], f["k"]

    # order clusters by their dominant task, then units within a cluster by peak TV
    n_k = labels.max() + 1
    dominant = np.array([X[labels == c].mean(axis=0).argmax() for c in range(n_k)])
    rank = np.empty(n_k, int); rank[np.argsort(dominant, kind="stable")] = np.arange(n_k)
    lab = rank[labels]
    order = np.lexsort((-X.max(axis=1), lab))

    inner = GridSpecFromSubplotSpec(3, 1, subplot_spec=spec[0], height_ratios=[18, 1.1, 1.1],
                                    hspace=0.08)
    ax = fig.add_subplot(inner[0])
    im = ax.imshow(X[order].T, aspect="auto", cmap="hot", vmin=0, vmax=1, interpolation="nearest")
    ax.set_yticks(range(len(TASKS)))
    ax.set_yticklabels([rule_name[t] for t in TASKS], fontsize=6.4, color=MUTED)
    ax.set_xticks([])
    m = ck[(p, s)]["metrics"]
    ax.set_title(f"{p}  ·  {POINTS[p][1]}      acc {m['min_task_acc']:.2f} · "
                 f"metabolic {m['metabolic_cost']:.3g} · conn_frac {m['conn_frac']:.3f}",
                 fontsize=9.5, color=INK, loc="left", pad=5)
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.tick_params(length=0)

    cmap_k = ListedColormap(plt.get_cmap("tab20")(np.linspace(0, 1, 20))[:max(n_k, 1)])
    ax_c = fig.add_subplot(inner[1])
    ax_c.imshow(lab[order][None, :], aspect="auto", cmap=cmap_k, vmin=-0.5, vmax=n_k - 0.5,
                interpolation="nearest")
    for c in range(n_k):
        idx = np.where(lab[order] == c)[0]
        if len(idx) > 4:
            ax_c.text(idx.mean(), 0, str(c + 1), ha="center", va="center", fontsize=6)
    ax_c.set_xticks([]); ax_c.set_yticks([0])
    ax_c.set_yticklabels([f"cluster (k={k})"], fontsize=6.6, color=MUTED)
    ax_c.tick_params(length=0)

    ax_e = fig.add_subplot(inner[2])
    is_e = (np.arange(len(f["active"])) < N_E)[f["active"]][order]
    ax_e.imshow(is_e[None, :], aspect="auto", cmap=ListedColormap(["#eb6834", "#2a78d6"]),
                vmin=0, vmax=1, interpolation="nearest")
    ax_e.set_xticks([]); ax_e.set_yticks([0])
    ax_e.set_yticklabels(["E / I"], fontsize=6.6, color=MUTED)
    ax_e.set_xlabel("units, sorted by cluster", fontsize=7.6, color=MUTED)
    ax_e.tick_params(length=0)

    axs = fig.add_subplot(spec[1])
    ks = list(K_RANGE)
    axs.fill_between(ks, [f["null"][k][0] for k in ks], [f["null"][k][1] for k in ks],
                     color="#bdbcb5", alpha=0.55, lw=0, label="shuffled null")
    axs.plot(ks, [f["scores"][k] for k in ks], color="#2a78d6", lw=2.0, marker="o", ms=3.6,
             mec=SURF, mew=0.6, label="observed")
    axs.axhline(SIL_REF, color=INK, ls="--", lw=1.2, alpha=0.8)
    axs.text(ks[-1], SIL_REF + 0.006, f"{SIL_REF}", fontsize=7.2, color=INK, ha="right", va="bottom")
    axs.plot([k], [f["scores"][k]], "o", ms=10, mfc="none", mec=ALERT, mew=2.0)
    axs.annotate(f"k = {k}\ns = {f['scores'][k]:.3f}", (k, f["scores"][k]), xytext=(8, 8),
                 textcoords="offset points", fontsize=7.6, color=ALERT)
    axs.set_xlabel("k", fontsize=8.2, color=MUTED)
    axs.set_ylabel("silhouette", fontsize=8.2, color=MUTED)
    axs.set_xticks([2, 5, 10, 15, 19])
    axs.set_ylim(0, max(0.42, max(f["scores"].values()) * 1.18))
    axs.grid(True, alpha=0.22, lw=0.7)
    axs.tick_params(labelsize=7.4, colors=MUTED, length=0)
    for sp in ("top", "right"): axs.spines[sp].set_visible(False)
    for sp in ("left", "bottom"): axs.spines[sp].set_color("#d8d7d0")
    if p == list(POINTS)[0]:
        axs.legend(fontsize=7, frameon=False, loc="upper right", labelcolor=MUTED)


def seed_figure(s):
    fig = plt.figure(figsize=(14.5, 15.0), facecolor=SURF)
    outer = fig.add_gridspec(4, 2, width_ratios=[2.5, 1], hspace=0.30, wspace=0.17,
                             left=0.085, right=0.975, top=0.945, bottom=0.045)
    for i, p in enumerate(POINTS):
        panel_row(fig, [outer[i, 0], outer[i, 1]], p, s)
    fig.suptitle(f"Task variance and clustering — frozen $W_{{out}}$ · cog20 · 150k · "
                 f"iso-accuracy contour · seed {s}",
                 fontsize=14, color=INK, x=0.085, ha="left", y=0.975)
    return fig

fig0 = seed_figure(0)
plt.show()

## Seed 1

In [ ]:
fig1 = seed_figure(1)
plt.show()